# Week 31, Day 4: Callable objects

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w031d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, x):
        return x * self.factor

double = Multiplier(2)
print(double(21), list(map(double, [1, 2, 3])), double.factor)
print(callable(double), callable(Multiplier), callable(42), callable(print))

In [ ]:
class Above:
    def __init__(self, limit):
        self.limit = limit

    def __call__(self, value):
        return value > self.limit

    def __repr__(self):
        return f"Above({self.limit})"

temps = [14.5, 22.1, 30.4, 18.0, 27.9]
hot = Above(25)
print(hot, list(filter(hot, temps)), hot(26), sum(map(Above(20), temps)))

In [ ]:
def make_above(limit):
    def check(value):
        return value > limit
    return check

class Above:
    def __init__(self, limit):
        self.limit = limit

    def __call__(self, value):
        return value > self.limit

f, g = make_above(25), Above(25)
print(f(30), g(30), [f(t) == g(t) for t in [10, 25, 26]])
print(g.limit)
g.limit = 29
print(g(27))

In [ ]:
class RunningTotal:
    def __init__(self):
        self.total = 0
        self.calls = 0

    def __call__(self, amount):
        self.total += amount
        self.calls += 1
        return self.total

    def reset(self):
        self.total = 0
        self.calls = 0

till = RunningTotal()
print([till(p) for p in [250, 120, 75]])
print(till.total, till.calls)
till.reset()
print(till(500))

In [ ]:
class Limiter:
    def __init__(self, max_calls, seconds):
        self.max_calls = max_calls
        self.seconds = seconds
        self.recent = []

    def __call__(self, now):
        self.recent = [t for t in self.recent if t > now - self.seconds]
        if len(self.recent) < self.max_calls:
            self.recent.append(now)
            return True
        return False

allow = Limiter(2, 10)
print([allow(t) for t in [0, 1, 2, 9, 10.5, 11, 12]])

In [ ]:
class Remember:
    def __init__(self, func):
        self.func = func
        self.saved = {}
        self.calls = 0

    def __call__(self, *args):
        self.calls += 1
        if args not in self.saved:
            self.saved[args] = self.func(*args)
        return self.saved[args]

def ways(n):
    """Ways to climb n stairs taking 1 or 2 at a time."""
    if n <= 1:
        return 1
    return ways(n - 1) + ways(n - 2)

ways = Remember(ways)
print(ways(30), ways.calls, len(ways.saved))

In [ ]:
class PercentOff:
    def __init__(self, percent):
        self.percent = percent

    def __call__(self, pence):
        return pence - pence * self.percent // 100

    def __repr__(self):
        return f"PercentOff({self.percent})"

def full_price(pence):
    return pence

def checkout(items, pricing=full_price):
    return sum(pricing(p) for p in items)

basket = [2400, 300, 1250]
print(checkout(basket), checkout(basket, PercentOff(10)), checkout(basket, lambda p: min(p, 1000)))

> **Key idea.** A class with `__call__` makes objects you can call like functions; `callable(x)` tests for it, and functions, methods, lambdas, classes and callable objects can all be passed wherever a callable is expected. Compared with a closure, a callable object exposes its settings, can change state between calls with `self`, and can have extra methods and a helpful repr. Keep state on `self`, not in class attributes. A callable object can wrap a function to add behaviour such as counting or caching. If something has no settings and no state, a plain function is better.

### Exercise: A VAT adder

Write `AddVat(rate)`, a callable object: calling it with a price in pence returns the price plus VAT at `rate` per cent, with the VAT rounded down to a whole penny (`pence * rate // 100`). `repr` gives `AddVat(20)`, and changing `.rate` changes later results.

In [ ]:
class AddVat:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_vat():\n    \"adding VAT\"\n    A = ns[\"AddVat\"]\n    v = A(20)\n    assert v(1000) == 1200 and v(999) == 1198 and list(map(v, [0, 5, 10])) == [0, 6, 12]\n    assert repr(v) == \"AddVat(20)\" and callable(v)\n    v.rate = 5\n    assert v(1000) == 1050 and A(0)(777) == 777")

### Exercise: A range check for filter

Write `Between(low, high)`, a callable that returns True for values from `low` to `high` **inclusive**. `low` greater than `high` raises `ValueError`. `repr` gives `Between(18, 25)`. Then use it to write `young_adults(ages)`, returning the ages from 18 to 25 with `filter`, as a list.

In [ ]:
class Between:
    ___

def young_adults(ages):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"checks\"\n    B = ns[\"Between\"]\n    b = B(18, 25)\n    assert b(18) and b(25) and b(21.5) and not b(17) and not b(26) and repr(b) == \"Between(18, 25)\"\n    assert young_adults([16, 18, 30, 25, 22, 26]) == [18, 25, 22] and young_adults([]) == []\n    assert \"filter\" in source\n    try:\n        B(5, 1)\n        assert False, \"low > high should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: A running average

Write `RunningMean()`: each call with a number adds it and returns the mean of all the numbers so far, rounded to 2 places. It has `count` (how many numbers so far) and `reset()`, which starts again from nothing. Each `RunningMean` keeps its own numbers. Don't keep a list of every number: a count and a total are enough.

In [ ]:
class RunningMean:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_mean():\n    \"averages\"\n    R = ns[\"RunningMean\"]\n    m = R()\n    assert [m(x) for x in [10, 20, 0, 5]] == [10, 15, 10, 8.75] and m.count == 4\n    assert m(1) == 7.2\n    other = R()\n    assert other(3) == 3 and m.count == 5, \"each RunningMean keeps its own numbers\"\n    m.reset()\n    assert m.count == 0 and m(7) == 7 and R()(2 / 3) == 0.67")

### Exercise: Remembering results

Write `Memo(func)`, a callable wrapper that remembers results by their positional arguments, with `hits` (calls answered from memory) and `misses` (calls that ran `func`), and `clear()`, which forgets every saved result (the counts stay). Then wrap `routes`, which counts the routes through a grid going only right or down, by writing `routes = Memo(routes)` after it, so its recursive calls are remembered too.

In [ ]:
class Memo:
    ___

def routes(rows, cols):
    if rows == 0 or cols == 0:
        return 1
    return routes(rows - 1, cols) + routes(rows, cols - 1)

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_memo():\n    \"remembering\"\n    M = ns[\"Memo\"]\n    seen = []\n    def square(x):\n        seen.append(x)\n        return x * x\n    sq = M(square)\n    assert [sq(3), sq(4), sq(3), sq(3)] == [9, 16, 9, 9] and seen == [3, 4]\n    assert (sq.hits, sq.misses) == (2, 2)\n    sq.clear()\n    assert sq(3) == 9 and seen == [3, 4, 3] and sq.misses == 3\ndef test_routes():\n    \"the grid\"\n    assert type(routes).__name__ == \"Memo\", \"wrap routes with routes = Memo(routes)\"\n    assert routes(16, 16) == 601080390 and routes(2, 2) == 6\n    assert routes.misses < 400, \"the recursive calls should go through the wrapper\"")

### Exercise: A weighted scorer for sorting

A college ranks applicants by a weighted score. Write `Weighted(**weights)`, a callable: called with a dictionary of an applicant's marks, it returns the sum of each weight times that mark (a missing mark counts as 0, and marks without a weight are ignored), rounded to 1 place. `repr` gives the weights in the order given: `Weighted(maths=2, english=1)`. Then write `ranking(applicants, scorer)`: the applicants' names, best score first, ties by name A to Z. `applicants` is a dictionary from name to marks. `ranking` must work with **any** callable scorer, including a plain function.

In [ ]:
class Weighted:
    ___

def ranking(applicants, scorer):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_weighted():\n    \"scores\"\n    W = ns[\"Weighted\"]\n    w = W(maths=2, english=1)\n    assert w({\"maths\": 70, \"english\": 60}) == 200 and w({\"maths\": 50}) == 100 and w({\"art\": 90}) == 0\n    assert W(maths=0.5)({\"maths\": 33}) == 16.5 and repr(w) == \"Weighted(maths=2, english=1)\"\ndef test_ranking():\n    \"rankings\"\n    W = ns[\"Weighted\"]\n    apps = {\"Ada\": {\"maths\": 80, \"english\": 50}, \"Bo\": {\"maths\": 60, \"english\": 90}, \"Chi\": {\"maths\": 75, \"english\": 70}}\n    assert ranking(apps, W(maths=2, english=1)) == [\"Chi\", \"Ada\", \"Bo\"], \"Ada and Bo tie on 210, so Ada comes first by name\"\n    assert ranking(apps, W(maths=1, english=1)) == [\"Bo\", \"Chi\", \"Ada\"]\n    assert ranking(apps, W(english=1)) == [\"Bo\", \"Chi\", \"Ada\"]\n    assert ranking(apps, lambda marks: 0) == [\"Ada\", \"Bo\", \"Chi\"] and ranking({}, W()) == []")

### Bug hunt: Missing self

Calling the converter crashes with `TypeError: Converter.__call__() takes 1 positional argument but 2 were given`. Fix it so it prints `[0.0, 1.61, 160.93]`.

In [ ]:
class Converter:
    def __init__(self, factor):
        self.factor = factor

    def __call__(value):
        return round(value * self.factor, 2)

miles_to_km = Converter(1.609344)
print([miles_to_km(m) for m in [0, 1, 100]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"converted\"\n    expect_output(\"[0.0, 1.61, 160.93]\")")

### Bug hunt: The bell rings too early

The bells should ring **after** `finished` is printed, but `ding` comes first and then the program crashes with `TypeError: 'NoneType' object is not callable`. Fix it so it prints `finished`, then `ding`, then `dong`.

In [ ]:
class Bell:
    def __init__(self, sound):
        self.sound = sound

    def __call__(self):
        print(self.sound)

when_done = []
when_done.append(Bell("ding")())
when_done.append(Bell("dong"))

print("finished")
for action in when_done:
    action()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"in order\"\n    expect_output(\"finished\\nding\\ndong\")")

### Bug hunt: Counters that share

Each turnstile should count its own people, so this should print `2 1`, but it prints `3 3`. Fix `Turnstile`.

In [ ]:
class Turnstile:
    count = 0

    def __call__(self):
        Turnstile.count += 1
        return Turnstile.count

north, south = Turnstile(), Turnstile()
north()
north()
south()
print(north.count, south.count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separate counts\"\n    expect_output(\"2 1\")\n    t = ns[\"Turnstile\"]()\n    assert t() == 1 and t() == 2")

### Puzzle: A greeter you can call

Arrange the lines so that `Greeter` objects can be called with a name. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Add:
    def __init__(self, n):
        self.n = n
    def __call__(self, x):
        return x + self.n

class Tally:
    def __init__(self):
        self.seen = 0
    def __call__(self):
        self.seen += 1
        return self.seen

a = Add(3)
t = Tally()
t()
t()

p = ___   # a(4)
q = ___   # list(map(Add(10), [1, 2]))
r = ___   # t()
s = ___   # callable(t), callable(t.seen)
u = ___   # sorted([3, -5, 1], key=abs)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (7, [11, 12], 3), \"p to r\"\n    assert s == (True, False) and u == [1, 3, -5]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a text pipeline

Build a pipeline that tidies text by passing it through steps, where a step is **any callable** that takes a string and returns a string.

- Step classes, each callable and each with a `repr` as shown: `Strip()` (removes spaces at both ends), `Lower()` (lower case), `Squash()` (turns every run of whitespace into a single space), `Replace(old, new)` (as `str.replace`; repr `Replace('&', 'and')`), and `Truncate(n)` (keeps the first `n` characters; if it cut anything, the last of those `n` characters becomes `…`; `n` below 1 raises `ValueError`; repr `Truncate(10)`).
- `Pipeline(*steps)` is itself callable: it passes the text through each step in order and returns the result. A step that isn't callable raises `TypeError` when the pipeline is made.
- `len(p)` is the number of steps. `p.then(step)` returns a **new** pipeline with the step added at the end, leaving `p` unchanged. `p | q` returns a new pipeline with `p`'s steps followed by `q`'s.
- `repr(p)` lists the steps: `Pipeline(Strip(), Lower(), Truncate(10))`. A plain function step appears by its name (`slugify`), a lambda as `<lambda>`.
- Since a pipeline is callable, a pipeline can be a step in another pipeline.

In [ ]:
import re

___

class Pipeline:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_steps():\n    \"steps\"\n    St, L, Sq, R, T = (ns[n] for n in [\"Strip\", \"Lower\", \"Squash\", \"Replace\", \"Truncate\"])\n    assert St()(\"  hi  \") == \"hi\" and L()(\"HeLLo\") == \"hello\" and Sq()(\"a  b\\t\\n c\") == \"a b c\"\n    assert R(\"&\", \"and\")(\"fish & chips\") == \"fish and chips\" and repr(R(\"&\", \"and\")) == \"Replace('&', 'and')\"\n    assert T(5)(\"Lagos\") == \"Lagos\" and T(4)(\"Lagos\") == \"Lag\u2026\" and T(1)(\"ab\") == \"\u2026\" and repr(T(10)) == \"Truncate(10)\"\n    assert repr(St()) == \"Strip()\" and repr(Sq()) == \"Squash()\"\n    try:\n        T(0)\n        assert False, \"Truncate(0) should raise ValueError\"\n    except ValueError:\n        pass\ndef test_pipeline():\n    \"pipelines\"\n    P, St, L, Sq, T = (ns[n] for n in [\"Pipeline\", \"Strip\", \"Lower\", \"Squash\", \"Truncate\"])\n    p = P(St(), Sq(), L())\n    assert p(\"  Jollof   RICE  \") == \"jollof rice\" and len(p) == 3 and P()(\"same\") == \"same\"\n    longer = p.then(T(8))\n    assert longer(\"  Jollof   RICE  \") == \"jollof \u2026\" and len(p) == 3 and len(longer) == 4\n    assert repr(longer) == \"Pipeline(Strip(), Squash(), Lower(), Truncate(8))\"\n    def slugify(text):\n        return text.replace(\" \", \"-\")\n    both = p | P(slugify, lambda s: s + \"!\")\n    assert both(\" Fried  Plantain \") == \"fried-plantain!\" and len(both) == 5\n    assert repr(both) == \"Pipeline(Strip(), Squash(), Lower(), slugify, <lambda>)\", f\"got {both!r}\"\n    nested = P(p, T(6))\n    assert nested(\"  BIG   Day \") == \"big d\u2026\" and len(nested) == 2\n    try:\n        P(St(), \"lower\")\n        assert False, \"a string isn't callable: TypeError\"\n    except TypeError:\n        pass")

## Warm-up

`double` is an ordinary object (you can read its `factor`), but `double(21)` calls it like a function, because its class has `__call__`. `map` didn't mind either: it only needs something it can call. And `Multiplier` itself is callable too, which is what happens every time you make an object.

## Objects you can call

`x(args)` calls `type(x).__call__(x, args)`. That's all there is to it: `__call__` takes `self` plus whatever arguments the object should accept, and returns a result, like any method.

`callable(x)` says whether `x` can be called. These are all callable:

| Kind | Example | Calling it... |
|---|---|---|
| function | `len`, `def area(...)` | runs it |
| lambda | `lambda x: x * 2` | runs it |
| bound method | `"abc".upper`, `basket.add` | runs the method on that object |
| class | `Book`, `int`, `list` | makes a new object |
| object with `__call__` | `Multiplier(2)` | runs `__call__` |

Every one of these can be passed wherever Python expects "a function": `map`, `filter`, `sorted(key=...)`, `max(key=...)`, a callback list, a dictionary of actions. That's duck typing again: whoever calls it only needs it to be callable.

The `__repr__` is a small bonus over a lambda, whose repr is `<function <lambda> at 0x...>`. When a callable is stored somewhere and printed in a debug message, `Above(25)` says exactly what it does.

## Callable objects and closures

Week 7, Day 3 made configurable functions with closures:

Both remember a setting and both work wherever a function is wanted. Which to choose?

- **A closure** is shorter, and fine when the function only needs fixed settings.
- **A callable object** is better when callers need to **see** the settings (`g.limit`), **change** them, or when the function keeps **state that changes** between calls, or needs **extra methods** (`reset`, `report`) and a readable repr.

(A closure can change a remembered variable too, but only with an extra keyword, `nonlocal`, which Phase 5 covers. In an object, it's just `self.something += 1`.)

## State between calls

The real strength of a callable object is memory. Each call can update `self`, and the next call sees the change:

Each object has its own state, so two tills keep separate totals. Make sure the state lives on `self` (set in `__init__`), not in a class attribute, which every object would share. You'll fix exactly that bug in the practice.

Here's a more practical one: a **rate limiter**, which allows at most a few actions in any period of time, as websites do to stop someone sending a hundred messages a second. Times are passed in, which keeps it easy to test:

At time 2 there have already been two calls in the last ten seconds, so it's refused; by 10.5, the call at 0 is more than ten seconds old and drops out.

## Wrapping a function

A callable object can hold **another** function and add something each time it's called. This wrapper counts calls and remembers results, so a slow function never computes the same answer twice:

- `*args` collects the positional arguments into a tuple (Week 7), which is hashable, so it can be a dictionary key.
- The line `ways = Remember(ways)` replaces the name `ways` with the wrapper. The recursive calls inside the original function look up `ways` by name, so they go through the wrapper too, and each step is worked out once: 59 calls instead of more than two million.
- This is the idea behind `functools.cache`, which you've used with `@cache` since Week 16. Writing `@cache` above a function does the same as `ways = cache(ways)` afterwards; Phase 5 explains how decorators like it work.

## Anything callable will do

Because callers only need something callable, code that takes "a function" accepts all the kinds in the table above. Week 30's pricing strategies had a `price` method; they could just as well be callables, and then a plain function works as a strategy too:

Classes are callables too, which makes them handy as **factories**: a dictionary from names to classes picks which kind of object to make (as Week 30's VAT challenge did), and `type(self)(...)` makes another object of the same class.

So when should something be a class with `__call__`, and when just a function? If it has no settings and no state, write a **function**: a one-method class with nothing in `__init__` is just a function with extra steps. Reach for a callable object when it needs settings worth inspecting, state between calls, or more than one method.

## Practice

## Quiz

## Daily challenge

## Before you go

Look for a lambda or closure in your own code that you've had to print while debugging, or that you wished you could adjust after making it, and turn it into a callable object with a helpful repr. Tomorrow: `__slots__` and memory, for classes that make millions of small objects.